[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/studiobuilders/ism-cyberrag/blob/main/notebooks/sprint1_poc.ipynb)

# ISM-CyberRAG — Sprint 1 POC

**Baseline RAG pipeline** for the Australian Information Security Manual.

Pipeline: Parse ISM PDFs → Chunk → Embed → Store in Supabase pgvector → Retrieve → Generate Answer (Groq) → Evaluate (RAGAS) → Log (ClearML)


---

**Team Name:** Studio Builders

**Team Members:**

- Sreekar Reddy Edulapalli (25617806)
- Chandan Sreenivasaiah (25674250)
- Ruben Easo Thomas (25598184)


---
## 1 · Environment Setup

In [2]:
# ── Set up the environment, clone repository, and install necessary dependencies ──
import os
import sys

# ── Clone repo if running in Colab / SageMaker ──
if not os.path.exists('ism-cyberrag') and not os.path.exists('../src'):
    !git clone -q https://github.com/studiobuilders/ism-cyberrag.git
    %cd ism-cyberrag

# ── Find project root and make src importable ──
cwd = os.getcwd()
if 'notebooks' in cwd:
    project_root = os.path.abspath('..')
else:
    project_root = cwd

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# ── Install dependencies ──
req_path = os.path.join(project_root, 'requirements.txt')
!pip install -qr {req_path}

print(f'Project root: {project_root}')
print('Environment ready.')


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Project root: /Users/sreekarreddy/Desktop/AIStudio/ism-cyberrag
Environment ready.


---
## 2 · ClearML Initialization

ClearML reads credentials from environment variables automatically (`CLEARML_API_ACCESS_KEY`, `CLEARML_API_SECRET_KEY`, `CLEARML_API_HOST`, `CLEARML_WEB_HOST`, `CLEARML_FILES_HOST`). Set these in your `.env` file.

In [ ]:
# ── Initialize ClearML to track data processing experiments ──
from src.config import CLEARML_PROJECT, CLEARML_TASK
from clearml import Task

task = Task.init(
    project_name=CLEARML_PROJECT,
    task_name=CLEARML_TASK,
    task_type=Task.TaskTypes.data_processing,
)
print(f'ClearML task created: {CLEARML_PROJECT} / {CLEARML_TASK}')
print(f'View at: https://app.clear.ml')

---
## 3 · Initialize Clients & Models

In [ ]:
# ── Initialize Supabase database client and load embedding models ──
from src.config import (
    EMBEDDING_MODEL_NAME, EMBEDDING_DIMENSION,
    CHUNK_SIZE, CHUNK_OVERLAP, MATCH_COUNT,
    DATA_DIR, EVAL_DIR, EVAL_DATASET_PATH,
)
from src.supabase_utils import get_supabase_client, count_rows
from src.embeddings import load_embedding_model

# ── Supabase ──
supabase = get_supabase_client()
print(f'Supabase client ready.')
print(f'  Existing chunks in DB: {count_rows(supabase, "chunks")}')

# ── Embedding Model ──
embedding_model = load_embedding_model()
print(f'Embedding model loaded: {EMBEDDING_MODEL_NAME} ({EMBEDDING_DIMENSION}-dim)')

---
## 4 · Data Ingestion Pipeline

Parse all 25 ISM PDFs → chunk → embed → store in Supabase pgvector.

> ⚠️ **Run this section once** to populate the database. Skip to Section 5 if data is already ingested.

### 4.1 · Parse PDFs

In [ ]:
# ── Parse the contents of all ISM PDF documents into text ──
from src.parse_pdf import parse_all_pdfs

documents = parse_all_pdfs(DATA_DIR)
print(f'\nTotal characters: {sum(len(d["text"]) for d in documents):,}')

### 4.2 · Chunk Documents

In [ ]:
# ── Split the parsed documents into processable overlapping chunks ──
from src.chunking import chunk_text

all_chunks = []
for doc in documents:
    chunks = chunk_text(doc['text'], chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
    for chunk in chunks:
        all_chunks.append({
            'content': chunk,
            'metadata': {
                'source_file': doc['source_file'],
                'title': doc['title'],
            },
        })

print(f'Total chunks: {len(all_chunks)}')
print(f'Config: chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}')

### 4.3 · Generate Embeddings

In [ ]:
# ── Generate vector embeddings for every chunk ──
from src.embeddings import embed_texts

texts = [c['content'] for c in all_chunks]
embeddings = embed_texts(embedding_model, texts)

print(f'Generated {len(embeddings)} embeddings')
print(f'Embedding dimension: {len(embeddings[0])}')

### 4.4 · Store in Supabase pgvector

In [ ]:
# ── Store the generated chunks and their embeddings into Supabase pgvector in batches ──
from src.supabase_utils import insert_chunks

# Build rows for insertion
rows = []
for i, chunk_data in enumerate(all_chunks):
    rows.append({
        'content': chunk_data['content'],
        'embedding': embeddings[i],
        'metadata': chunk_data['metadata'],
    })

# Insert in batches
BATCH_SIZE = 50
for start in range(0, len(rows), BATCH_SIZE):
    batch = rows[start:start + BATCH_SIZE]
    insert_chunks(supabase, batch)
    print(f'  Batch {start // BATCH_SIZE + 1}: inserted {len(batch)} rows')

total = count_rows(supabase, 'chunks')
print(f'\nTotal rows in chunks table: {total}')

---
## 5 · Retrieval Demo

Test vector similarity search on a sample question.

In [ ]:
# ── Perform a test vector similarity search with a sample question ──
from src.embeddings import embed_query
from src.retrieval import match_chunks

question = 'What does the ISM say about multi-factor authentication?'

query_emb = embed_query(embedding_model, question)
results = match_chunks(supabase, query_emb, match_count=MATCH_COUNT)

print(f'Question: {question}\n')
for i, r in enumerate(results, 1):
    sim = r.get('similarity', 0)
    print(f'--- Chunk {i} (similarity: {sim:.4f}) ---')
    print(r['content'][:300])
    print()

---
## 6 · Answer Generation

Generate a cited answer using Groq (Llama 3.1 8B).

In [ ]:
# ── Generate the final answer using the Groq LLM model ──
from src.llm import generate_answer

answer = generate_answer(question, results)

print(f'Question: {question}\n')
print(f'Answer:\n{answer}')

---
## 7 · Interactive Q&A Helper

Convenience function for end-to-end RAG.

In [ ]:
# ── Interactive Q&A Helper function for end-to-end RAG testing ──
def ask_ism(question: str, top_k: int = MATCH_COUNT):
    """End-to-end: embed query → retrieve → generate answer."""
    q_emb = embed_query(embedding_model, question)
    chunks = match_chunks(supabase, q_emb, match_count=top_k)
    answer = generate_answer(question, chunks)
    return answer, chunks

# Test it
q = 'What does the ISM say about multi-factor access?'
ans, ctx = ask_ism(q)
print(f'Q: {q}\n')
print(f'A: {ans}')

---
## 8 · RAGAS Evaluation

Evaluate the baseline pipeline on the 20-question dataset.

In [ ]:
# ── Load the eval dataset to evaluate pipeline on 20 questions ──
import pandas as pd
from src.evaluation import load_eval_dataset, run_ragas_evaluation

eval_data = load_eval_dataset(EVAL_DATASET_PATH)
pd.DataFrame(eval_data)[['question', 'category']]

In [ ]:
# ── Run the RAG pipeline on all evaluation questions using retrieval and generation functions ──
# Define retrieve and generate functions for the evaluator
def retrieve_fn(question):
    q_emb = embed_query(embedding_model, question)
    return match_chunks(supabase, q_emb, match_count=MATCH_COUNT)

def generate_fn(question, chunks):
    return generate_answer(question, chunks)

# Run pipeline on all eval questions
print('Running RAG pipeline on evaluation dataset...\n')
eval_results = run_ragas_evaluation(eval_data, retrieve_fn, generate_fn)
print(f'\nCompleted {len(eval_results)} evaluations.')

In [ ]:
# ── Compute final RAGAS scores (e.g., context precision/recall, answer relevancy) using updated evaluation logic ──

from src.evaluation import compute_ragas_scores

ragas_result = compute_ragas_scores(eval_results)

---
## 9 · Log Results to ClearML

In [ ]:
# ── Log evaluation metrics and parameters back to ClearML dashboard ──
from src.config import LLM_MODEL_NAME
from src.evaluation import log_metrics_to_clearml

# Extract numeric metrics only
numeric_metrics = {k: v for k, v in ragas_result.items() if isinstance(v, (int, float))}

log_metrics_to_clearml(
    metrics=numeric_metrics,
    params={
        'chunk_size': CHUNK_SIZE,
        'chunk_overlap': CHUNK_OVERLAP,
        'embedding_model': EMBEDDING_MODEL_NAME,
        'embedding_dimension': EMBEDDING_DIMENSION,
        'llm_model': LLM_MODEL_NAME,
        'retrieval_method': 'vector_similarity',
        'match_count': MATCH_COUNT,
        'num_eval_questions': len(eval_data),
        'num_chunks_in_db': count_rows(supabase, 'chunks'),
    }
)

print('All metrics and parameters logged to ClearML.')

---
## 10 · Close ClearML Task

In [ ]:
# ── Terminate and mark the current ClearML tracking task as completed ──
from clearml import Task

# Get the currently running task and close it
current_task = Task.current_task()
if current_task:
    current_task.close()
    print("✓ ClearML task successfully closed and marked as 'Completed'.")
else:
    print("No active ClearML task found.")